# Prepare a YOLOv8-OBB License-Plate Blur Dataset — Intel Arc / PyTorch XPU Version

This notebook creates aligned clean, motion-blurred, and defocus-blurred license-plate datasets from **YOLOv8 OBB four-corner annotations**.

**Intel GPU behavior:**
- OpenCV still reads/writes image files on the CPU.
- OBB perspective rectification runs with **PyTorch XPU** on an Intel GPU.
- Motion-blur and defocus-blur convolution runs with **PyTorch XPU**.
- `REQUIRE_XPU = True` makes the notebook stop instead of silently falling back to CPU.

This version is intended for Intel Arc GPUs such as your **Intel Arc A750**. The staged workflow remains: validate → inspect OBBs/crops → preview blur → generate full dataset → verify outputs.


## 1. Project Overview

Expected input structure:

```text
dataset/
├── clean_image/
│   ├── image001.jpg
│   ├── image002.jpg
│   └── ...
└── labels/
    ├── image001.txt
    ├── image002.txt
    └── ...
```

Each non-empty annotation line must be YOLOv8 OBB four-corner format:

```text
class_id x1 y1 x2 y2 x3 y3 x4 y4
```

The eight coordinates are normalized to `[0, 1]` and are converted to floating-point pixel coordinates before perspective extraction.

## 2. Install / Import Libraries

Create/activate the same VS Code virtual environment that the notebook kernel will use.

Install the normal dependencies once:

```bash
pip install opencv-python numpy matplotlib tqdm pandas jupyter ipykernel
```

Install the **Intel XPU build of PyTorch** in that same environment:

```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/xpu
```

Verify Intel GPU access from the VS Code terminal:

```bash
python -c "import torch; print(torch.__version__); print('XPU available:', torch.xpu.is_available()); print('XPU count:', torch.xpu.device_count()); print(torch.xpu.get_device_properties(0) if torch.xpu.is_available() else 'NO XPU')"
```

For this notebook you want `XPU available: True`. You do **not** need NVIDIA CUDA, `nvidia-smi`, or a CUDA-enabled OpenCV build.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import math
import os
import random
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

XPU_API_PRESENT = hasattr(torch, 'xpu')
XPU_AVAILABLE = bool(XPU_API_PRESENT and torch.xpu.is_available())

print('Python image-processing environment loaded successfully.')
print('OpenCV version :', cv2.__version__)
print('NumPy version  :', np.__version__)
print('Pandas version :', pd.__version__)
print('PyTorch version:', torch.__version__)
print('XPU API present:', XPU_API_PRESENT)
print('XPU available  :', XPU_AVAILABLE)

if XPU_AVAILABLE:
    print('XPU device count:', torch.xpu.device_count())
    for i in range(torch.xpu.device_count()):
        props = torch.xpu.get_device_properties(i)
        print(f'  XPU {i}: {props.name} | {props.total_memory / 1024**3:.2f} GB | driver {props.driver_version}')
else:
    print('No Intel XPU device is currently available to this PyTorch environment.')


In [ ]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("XPU available:", torch.xpu.is_available())
print("XPU count:", torch.xpu.device_count())

if torch.xpu.is_available():
    print("GPU:", torch.xpu.get_device_properties(0).name)


## 3. Configuration

Normally you only need to change `DATASET_ROOT`.

For Intel GPU processing, leave `REQUIRE_XPU = True`. The notebook will stop with a clear error if the selected Jupyter environment cannot access PyTorch XPU, rather than silently doing the expensive transformations on the CPU.


In [ ]:
# ============================================================
# PATHS, INTEL XPU & GENERAL SETTINGS
# ============================================================

# CHANGE THIS PATH to your dataset root.
# Example for your project:
# DATASET_ROOT = Path(r'D:\path\to\LicensePlateBlurProject\dataset')  # TODO: set your local dataset path
DATASET_ROOT = Path(r'D:\academy\thesis\dataset\thesis dataset\yolov8m-obb\input')  # TODO: set your local dataset path

IMAGE_DIR = DATASET_ROOT / 'clean_image'
LABEL_DIR = DATASET_ROOT / 'labels'

OUTPUT_ROOT = DATASET_ROOT / 'processed_dataset'
CLEAN_OUTPUT = OUTPUT_ROOT / 'clean_plate'
MOTION_OUTPUT = OUTPUT_ROOT / 'motionblurr_plate'
DEFOCUS_OUTPUT = OUTPUT_ROOT / 'defocusblurr_plate'
REPORT_PATH = OUTPUT_ROOT / 'processing_report.csv'

# ------------------------------
# INTEL XPU SETTINGS
# ------------------------------
REQUIRE_XPU = True      # Recommended for this Intel-GPU notebook.
XPU_INDEX = 0           # Your Arc A750 is normally xpu:0.

XPU_API_PRESENT = hasattr(torch, 'xpu')
XPU_AVAILABLE = bool(XPU_API_PRESENT and torch.xpu.is_available())

if XPU_AVAILABLE:
    xpu_count = torch.xpu.device_count()
    if XPU_INDEX < 0 or XPU_INDEX >= xpu_count:
        raise ValueError(
            f'XPU_INDEX={XPU_INDEX} is invalid. Available XPU devices: 0..{xpu_count - 1}'
        )
    DEVICE = torch.device(f'xpu:{XPU_INDEX}')
    torch.xpu.set_device(XPU_INDEX)
else:
    DEVICE = torch.device('cpu')

if REQUIRE_XPU and DEVICE.type != 'xpu':
    raise RuntimeError(
        'Intel XPU GPU is required but PyTorch cannot access it.\n'
        '1) Confirm the Intel Arc driver is installed and current.\n'
        '2) Activate the same .venv used by this notebook.\n'
        '3) Install PyTorch XPU: pip install torch torchvision --index-url https://download.pytorch.org/whl/xpu\n'
        '4) Restart the Jupyter kernel.\n'
        '5) Confirm: import torch; print(torch.xpu.is_available()) -> True'
    )

SUPPORTED_IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
OUTPUT_EXTENSION = '.jpg'       # Consistently save generated plates as JPEG.
JPEG_QUALITY = 95
OVERWRITE_EXISTING = False      # Never deletes output directories.
AUTO_ROTATE_LANDSCAPE = True    # Rotate rectified crop by 90 degrees if height > width.

# OBB/crop validation thresholds.
COORD_TOLERANCE = 1e-3
MIN_QUAD_AREA = 16.0            # pixels^2
MIN_CROP_WIDTH = 8              # pixels
MIN_CROP_HEIGHT = 4             # pixels

# Preview sizes.
OBB_PREVIEW_COUNT = 8
CROP_PREVIEW_COUNT = 8
BLUR_PREVIEW_COUNT = 8
FINAL_PREVIEW_COUNT = 10

print('=' * 68)
print('CONFIGURATION')
print('=' * 68)
print('DATASET_ROOT :', DATASET_ROOT)
print('Compute device:', DEVICE)
if DEVICE.type == 'xpu':
    props = torch.xpu.get_device_properties(XPU_INDEX)
    print('GPU          :', props.name)
    print('GPU memory   :', f'{props.total_memory / 1024**3:.2f} GB')
    print('Driver       :', props.driver_version)
    print('Backend      : PyTorch XPU / Intel oneAPI Level Zero')
else:
    print('WARNING: using CPU because REQUIRE_XPU=False.')
print('=' * 68)


In [ ]:
# ============================================================
# BLUR SETTINGS - MODIFY THESE
# Rerun this cell, then rerun ONLY the preview cell(s) below.
# Full dataset regeneration is NOT required while tuning.
# ============================================================

RANDOM_SEED = 42

# Moderate default motion blur. Kernel size is auto-reduced for small crops.
MOTION_KERNEL_SIZE = 9
MOTION_ANGLE_MIN = -20.0
MOTION_ANGLE_MAX = 20.0

# Disk/circular point-spread-function radius for defocus blur.
DEFOCUS_RADIUS = 2

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if hasattr(torch, 'xpu') and torch.xpu.is_available():
    torch.xpu.manual_seed_all(RANDOM_SEED)

print('Blur settings:')
print('  RANDOM_SEED         =', RANDOM_SEED)
print('  MOTION_KERNEL_SIZE  =', MOTION_KERNEL_SIZE)
print('  MOTION_ANGLE_RANGE  =', (MOTION_ANGLE_MIN, MOTION_ANGLE_MAX))
print('  DEFOCUS_RADIUS      =', DEFOCUS_RADIUS)
print('  COMPUTE_DEVICE      =', DEVICE)

### Intel XPU smoke test

This tiny test checks the two GPU operations this notebook relies on most: `grid_sample` for perspective resampling and grouped `conv2d` for blur. It does not touch your dataset.


In [ ]:
def run_xpu_smoke_test(device=DEVICE):
    if device.type != 'xpu':
        print('XPU smoke test skipped because DEVICE is', device)
        return

    try:
        with torch.inference_mode():
            # Small synthetic RGB tensor on Intel GPU.
            x = torch.rand((1, 3, 32, 96), device=device, dtype=torch.float32)

            # Identity sampling grid: exercises the same backend used by perspective extraction.
            ys = torch.linspace(-1.0, 1.0, 32, device=device)
            xs = torch.linspace(-1.0, 1.0, 96, device=device)
            yy, xx = torch.meshgrid(ys, xs, indexing='ij')
            grid = torch.stack((xx, yy), dim=-1).unsqueeze(0)
            warped = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=True)

            # 3x3 depthwise blur: exercises grouped conv2d on XPU.
            kernel = torch.ones((3, 3), device=device, dtype=torch.float32) / 9.0
            weight = kernel.reshape(1, 1, 3, 3).repeat(3, 1, 1, 1)
            padded = F.pad(warped, (1, 1, 1, 1), mode='reflect')
            blurred = F.conv2d(padded, weight, groups=3)

            torch.xpu.synchronize(device)

        assert blurred.shape == x.shape
        print('Intel XPU smoke test: PASSED')
        print('Device:', device)
        print('GPU   :', torch.xpu.get_device_properties(XPU_INDEX).name)
        print('Test tensor shape:', tuple(blurred.shape))
    except Exception as exc:
        raise RuntimeError(
            'Intel XPU smoke test FAILED. The notebook will not be reliable until this is fixed. '
            f'Original error: {exc}'
        ) from exc

run_xpu_smoke_test()


## 4. Dataset Path Setup

This cell only creates output directories if necessary. It never deletes them. Input discovery is **non-recursive**, so generated folders are never reprocessed as source images.

In [ ]:
print('DATASET_ROOT :', DATASET_ROOT)
print('IMAGE_DIR    :', IMAGE_DIR)
print('LABEL_DIR    :', LABEL_DIR)
print('OUTPUT_ROOT  :', OUTPUT_ROOT)

if not IMAGE_DIR.exists():
    raise FileNotFoundError(f'Image directory does not exist: {IMAGE_DIR}')
if not LABEL_DIR.exists():
    raise FileNotFoundError(f'Label directory does not exist: {LABEL_DIR}')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CLEAN_OUTPUT.mkdir(parents=True, exist_ok=True)
MOTION_OUTPUT.mkdir(parents=True, exist_ok=True)
DEFOCUS_OUTPUT.mkdir(parents=True, exist_ok=True)

print('\nOutput directories are ready (nothing was deleted).')

## 5. YOLOv8 OBB Annotation Parser

The parser validates `class_id + 8 normalized coordinates`, converts normalized coordinates to pixel coordinates, and reports malformed lines without stopping the whole dataset.

In [ ]:
def read_image(image_path: Path):
    """Read an image robustly, including many Windows paths with non-ASCII characters."""
    try:
        data = np.fromfile(str(image_path), dtype=np.uint8)
        if data.size == 0:
            return None
        return cv2.imdecode(data, cv2.IMREAD_COLOR)
    except Exception:
        return None


def parse_yolo_obb_label(label_path: Path, image_width: int, image_height: int,
                         coord_tolerance: float = COORD_TOLERANCE):
    """
    Parse YOLOv8 OBB labels in the format:
        class_id x1 y1 x2 y2 x3 y3 x4 y4

    Returns
    -------
    annotations : list[dict]
        Each item contains class_id, 4x2 floating-point pixel points,
        source line number, and whether tiny numerical clipping occurred.
    issues : list[dict]
        Non-fatal parsing/validation problems.
    """
    annotations = []
    issues = []

    if not label_path.exists():
        issues.append({'status': 'missing_label', 'line': None,
                       'message': f'Label file not found: {label_path.name}'})
        return annotations, issues

    try:
        lines = label_path.read_text(encoding='utf-8-sig', errors='replace').splitlines()
    except Exception as exc:
        issues.append({'status': 'label_read_failed', 'line': None,
                       'message': f'Could not read label file: {exc}'})
        return annotations, issues

    nonempty = [(i + 1, line.strip()) for i, line in enumerate(lines) if line.strip()]
    if not nonempty:
        issues.append({'status': 'empty_label', 'line': None,
                       'message': 'Label file is empty.'})
        return annotations, issues

    for line_number, line in nonempty:
        parts = line.split()
        if len(parts) != 9:
            issues.append({
                'status': 'invalid_annotation',
                'line': line_number,
                'message': f'Expected 9 values (class + 8 coords), found {len(parts)}.'
            })
            continue

        try:
            class_value = float(parts[0])
            coords = np.asarray([float(v) for v in parts[1:]], dtype=np.float64)
        except ValueError as exc:
            issues.append({'status': 'invalid_annotation', 'line': line_number,
                           'message': f'Non-numeric value: {exc}'})
            continue

        if not np.isfinite(class_value) or not np.all(np.isfinite(coords)):
            issues.append({'status': 'invalid_annotation', 'line': line_number,
                           'message': 'Class/coordinate contains NaN or infinity.'})
            continue

        if class_value < 0 or not float(class_value).is_integer():
            issues.append({'status': 'invalid_annotation', 'line': line_number,
                           'message': f'Class ID must be a non-negative integer, got {class_value}.'})
            continue

        below = coords < -coord_tolerance
        above = coords > (1.0 + coord_tolerance)
        if np.any(below | above):
            issues.append({
                'status': 'invalid_annotation',
                'line': line_number,
                'message': (
                    'Normalized OBB coordinate outside expected [0, 1] range '
                    f'beyond tolerance {coord_tolerance}.'
                )
            })
            continue

        clipped = bool(np.any((coords < 0.0) | (coords > 1.0)))
        if clipped:
            coords = np.clip(coords, 0.0, 1.0)
            issues.append({
                'status': 'coordinate_clipped',
                'line': line_number,
                'message': 'Tiny out-of-range coordinate was clipped to [0, 1].'
            })

        points_norm = coords.reshape(4, 2)
        points_px = points_norm.copy()
        points_px[:, 0] *= float(image_width)
        points_px[:, 1] *= float(image_height)

        annotations.append({
            'class_id': int(class_value),
            'points': points_px.astype(np.float32),
            'line_number': line_number,
            'clipped': clipped,
        })

    return annotations, issues

## 6. Dataset Validation — Stage A

Checks input directories, counts, unsupported files, image/label pairing, duplicate image stems, empty labels, and malformed OBB annotation lines. Problems are reported, not silently corrected.

In [ ]:
def discover_input_files():
    """Discover source files non-recursively so generated outputs are never re-ingested."""
    all_entries = [p for p in IMAGE_DIR.iterdir() if p.is_file()]
    image_files = sorted(
        [p for p in all_entries if p.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS],
        key=lambda p: p.name.lower()
    )
    unsupported = sorted(
        [p for p in all_entries if p.suffix.lower() not in SUPPORTED_IMAGE_EXTENSIONS],
        key=lambda p: p.name.lower()
    )
    label_files = sorted(LABEL_DIR.glob('*.txt'), key=lambda p: p.name.lower())
    return image_files, label_files, unsupported


def build_stem_index(paths):
    index = defaultdict(list)
    for path in paths:
        index[path.stem].append(path)
    return index


def validate_dataset():
    image_files, label_files, unsupported = discover_input_files()
    image_index = build_stem_index(image_files)
    label_index = build_stem_index(label_files)

    image_stems = set(image_index)
    label_stems = set(label_index)
    matched_stems = sorted(image_stems & label_stems)
    missing_labels = sorted(image_stems - label_stems)
    missing_images = sorted(label_stems - image_stems)
    duplicate_image_stems = {k: v for k, v in image_index.items() if len(v) > 1}

    malformed_lines = 0
    empty_labels = 0
    label_read_failures = 0
    clipped_lines = 0
    malformed_examples = []

    # Structural validation can use width=height=1 because normalized-coordinate
    # validity is independent of actual image dimensions.
    for label_path in label_files:
        _, issues = parse_yolo_obb_label(label_path, 1, 1)
        for issue in issues:
            status = issue['status']
            if status == 'invalid_annotation':
                malformed_lines += 1
            elif status == 'empty_label':
                empty_labels += 1
            elif status == 'label_read_failed':
                label_read_failures += 1
            elif status == 'coordinate_clipped':
                clipped_lines += 1
            if status in {'invalid_annotation', 'empty_label', 'label_read_failed'} and len(malformed_examples) < 10:
                malformed_examples.append((label_path.name, issue.get('line'), issue['message']))

    summary = {
        'image_files': image_files,
        'label_files': label_files,
        'unsupported_files': unsupported,
        'image_index': image_index,
        'label_index': label_index,
        'matched_stems': matched_stems,
        'missing_labels': missing_labels,
        'missing_images': missing_images,
        'duplicate_image_stems': duplicate_image_stems,
        'malformed_lines': malformed_lines,
        'empty_labels': empty_labels,
        'label_read_failures': label_read_failures,
        'clipped_lines': clipped_lines,
    }

    print('=' * 68)
    print('STAGE A - DATASET VALIDATION')
    print('=' * 68)
    print(f'Image directory exists:          {IMAGE_DIR.exists()}')
    print(f'Label directory exists:          {LABEL_DIR.exists()}')
    print(f'Supported images found:          {len(image_files)}')
    print(f'Label files found:               {len(label_files)}')
    print(f'Matched image-label stems:       {len(matched_stems)}')
    print(f'Missing label files:             {len(missing_labels)}')
    print(f'Labels missing images:           {len(missing_images)}')
    print(f'Unsupported input files:         {len(unsupported)}')
    print(f'Duplicate image stems:           {len(duplicate_image_stems)}')
    print(f'Empty label files:               {empty_labels}')
    print(f'Malformed annotation lines:      {malformed_lines}')
    print(f'Label read failures:             {label_read_failures}')
    print(f'Tiny coordinate clips recorded:  {clipped_lines}')

    if missing_labels[:10]:
        print('\nFirst missing labels:', missing_labels[:10])
    if missing_images[:10]:
        print('First labels missing images:', missing_images[:10])
    if unsupported[:10]:
        print('First unsupported files:', [p.name for p in unsupported[:10]])
    if duplicate_image_stems:
        print('Duplicate image stems (first 10):')
        for stem, paths in list(duplicate_image_stems.items())[:10]:
            print(' ', stem, '->', [p.name for p in paths])
    if malformed_examples:
        print('\nMalformed/empty label examples (up to 10):')
        for name, line, message in malformed_examples:
            where = f' line {line}' if line is not None else ''
            print(f'  {name}{where}: {message}')

    print('=' * 68)
    return summary


VALIDATION = validate_dataset()

## 7. OBB Point Ordering

`order_points()` uses all four vertices, sorts them around the quadrilateral centroid, rotates the sequence to an approximate top-left start, and enforces the expected `TL, TR, BR, BL` winding. This avoids blindly trusting annotation order.

In [ ]:
def order_points(pts):
    """
    Return four 2D points approximately ordered as TL, TR, BR, BL.

    Strategy:
    1. Sort the four vertices cyclically around their centroid.
    2. Compare the two pairs of opposite edges. For a license plate, the pair
       with the larger average length is treated as the plate-width direction.
    3. Rotate/reverse the cyclic order so the upper long edge runs left-to-right.

    This is robust to arbitrary annotation-point order and also helps keep the
    rectified plate in a landscape orientation for strongly rotated plates.
    """
    pts = np.asarray(pts, dtype=np.float32)
    if pts.shape != (4, 2):
        raise ValueError(f'Expected points with shape (4, 2), got {pts.shape}.')
    if not np.all(np.isfinite(pts)):
        raise ValueError('OBB points contain NaN or infinity.')

    # Reject duplicate/near-duplicate vertices.
    pairwise = []
    for i in range(4):
        for j in range(i + 1, 4):
            pairwise.append(float(np.linalg.norm(pts[i] - pts[j])))
    if min(pairwise) < 1e-3:
        raise ValueError('OBB contains duplicate or nearly duplicate corner points.')

    center = pts.mean(axis=0)
    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
    cyclic = pts[np.argsort(angles)]

    edge_lengths = np.asarray([
        np.linalg.norm(cyclic[(i + 1) % 4] - cyclic[i]) for i in range(4)
    ], dtype=np.float32)

    # Opposite edge pairs are (0,2) and (1,3). Put the longer pair at edges 0 and 2.
    mean_pair_02 = float((edge_lengths[0] + edge_lengths[2]) / 2.0)
    mean_pair_13 = float((edge_lengths[1] + edge_lengths[3]) / 2.0)
    if mean_pair_13 > mean_pair_02:
        cyclic = np.roll(cyclic, -1, axis=0)

    # Of the two long edges, put the one with the smaller midpoint-y first (top edge).
    edge0_mid_y = float((cyclic[0, 1] + cyclic[1, 1]) / 2.0)
    edge2_mid_y = float((cyclic[2, 1] + cyclic[3, 1]) / 2.0)
    if edge0_mid_y > edge2_mid_y:
        cyclic = np.roll(cyclic, -2, axis=0)

    # Ensure the top edge runs left-to-right. Reverse winding while preserving edge pairing.
    if cyclic[0, 0] > cyclic[1, 0]:
        cyclic = cyclic[[1, 0, 3, 2]]

    ordered = cyclic.astype(np.float32)

    # Sanity checks: polygon must remain non-degenerate and convex enough for a warp.
    if abs(float(cv2.contourArea(ordered))) < 1e-6:
        raise ValueError('Ordered OBB is degenerate (near-zero area).')

    return ordered

## 8. Four-Point License-Plate Extraction — Intel XPU

The quadrilateral is still defined by all four YOLOv8 OBB corners. Geometry validation and creation/inversion of the tiny 3×3 homography are inexpensive CPU operations, while the actual perspective resampling is performed by **PyTorch `grid_sample` on the selected Intel XPU GPU**.


In [ ]:
def bgr_numpy_to_tensor(image, device=DEVICE):
    """Convert OpenCV BGR uint8 HxWxC image to float BCHW tensor on DEVICE."""
    if image is None or image.size == 0:
        raise ValueError('Cannot convert an empty image to a tensor.')
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    if image.ndim != 3 or image.shape[2] != 3:
        raise ValueError(f'Expected a 3-channel image, got shape {image.shape}.')

    arr = np.ascontiguousarray(image)
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    return tensor.to(device=device, dtype=torch.float32, non_blocking=True).div_(255.0)


def tensor_to_bgr_numpy(tensor):
    """Convert a BCHW/CHW float tensor in [0,1] to OpenCV BGR uint8 HxWxC."""
    if tensor.ndim == 4:
        if tensor.shape[0] != 1:
            raise ValueError(f'Expected batch size 1, got shape {tuple(tensor.shape)}.')
        tensor = tensor[0]
    if tensor.ndim != 3:
        raise ValueError(f'Expected CHW tensor, got shape {tuple(tensor.shape)}.')

    out = tensor.detach().clamp(0.0, 1.0).mul(255.0).round().to(torch.uint8)
    out = out.permute(1, 2, 0).contiguous().cpu().numpy()
    return out


def _xpu_warp_perspective(image_tensor, matrix_src_to_dst, output_width, output_height):
    """
    Perspective warp using torch.grid_sample on image_tensor.device (Intel XPU or CPU fallback).

    matrix_src_to_dst maps source pixel coordinates -> destination coordinates.
    grid_sample requires destination -> source sampling coordinates, so the
    homography is inverted on the same compute device.
    """
    if image_tensor.ndim != 4 or image_tensor.shape[0] != 1:
        raise ValueError('image_tensor must have shape (1,C,H,W).')

    _, _, src_h, src_w = image_tensor.shape
    device = image_tensor.device
    dtype = image_tensor.dtype

    # Invert the tiny 3x3 homography on CPU for broad XPU compatibility.
    # The expensive per-pixel resampling still runs on the Intel GPU.
    H_inv_np = np.linalg.inv(np.asarray(matrix_src_to_dst, dtype=np.float64)).astype(np.float32)
    H_inv = torch.as_tensor(H_inv_np, device=device, dtype=dtype)

    ys = torch.arange(output_height, device=device, dtype=dtype)
    xs = torch.arange(output_width, device=device, dtype=dtype)
    yy, xx = torch.meshgrid(ys, xs, indexing='ij')
    ones = torch.ones_like(xx)

    dst_homogeneous = torch.stack((xx, yy, ones), dim=-1).reshape(-1, 3)
    src_homogeneous = dst_homogeneous @ H_inv.T

    z = src_homogeneous[:, 2].clamp_min(1e-8)
    src_x = src_homogeneous[:, 0] / z
    src_y = src_homogeneous[:, 1] / z

    # align_corners=True pixel normalization: pixel 0 -> -1, last pixel -> +1.
    if src_w > 1:
        grid_x = 2.0 * src_x / float(src_w - 1) - 1.0
    else:
        grid_x = torch.zeros_like(src_x)
    if src_h > 1:
        grid_y = 2.0 * src_y / float(src_h - 1) - 1.0
    else:
        grid_y = torch.zeros_like(src_y)

    grid = torch.stack((grid_x, grid_y), dim=-1).reshape(1, output_height, output_width, 2)

    return F.grid_sample(
        image_tensor,
        grid,
        mode='bilinear',
        padding_mode='border',
        align_corners=True,
    )


def extract_plate_tensor_from_obb(image, points,
                                  image_tensor=None,
                                  min_quad_area=MIN_QUAD_AREA,
                                  min_crop_width=MIN_CROP_WIDTH,
                                  min_crop_height=MIN_CROP_HEIGHT,
                                  auto_rotate_landscape=AUTO_ROTATE_LANDSCAPE):
    """
    Perspective-rectify one OBB plate and keep the result on DEVICE.

    Pass image_tensor when processing multiple OBBs from the same source image
    so the source image is uploaded to the Intel GPU only once.
    """
    info = {'status': 'invalid_crop', 'message': '', 'compute_device': str(DEVICE)}

    if image is None or image.size == 0:
        info['message'] = 'Input image is empty.'
        return None, info

    try:
        rect = order_points(points)
    except Exception as exc:
        info['message'] = f'Point ordering failed: {exc}'
        return None, info

    tl, tr, br, bl = rect
    quad_area = float(abs(cv2.contourArea(rect.astype(np.float32))))
    info['quad_area'] = quad_area
    info['ordered_points'] = rect

    if quad_area < float(min_quad_area):
        info['message'] = f'OBB area too small: {quad_area:.2f} px^2.'
        return None, info

    width_top = float(np.linalg.norm(tr - tl))
    width_bottom = float(np.linalg.norm(br - bl))
    height_left = float(np.linalg.norm(bl - tl))
    height_right = float(np.linalg.norm(br - tr))

    max_width = int(round(max(width_top, width_bottom)))
    max_height = int(round(max(height_left, height_right)))

    info.update({
        'width_top': width_top,
        'width_bottom': width_bottom,
        'height_left': height_left,
        'height_right': height_right,
        'raw_output_width': max_width,
        'raw_output_height': max_height,
    })

    if max_width <= 0 or max_height <= 0:
        info['message'] = f'Perspective destination has invalid size {max_width}x{max_height}.'
        return None, info
    if max_width < min_crop_width or max_height < min_crop_height:
        info['message'] = (
            f'Crop too small before warp: {max_width}x{max_height}; '
            f'minimum is {min_crop_width}x{min_crop_height}.'
        )
        return None, info

    dst = np.asarray([
        [0.0, 0.0],
        [max_width - 1.0, 0.0],
        [max_width - 1.0, max_height - 1.0],
        [0.0, max_height - 1.0],
    ], dtype=np.float32)

    try:
        # The 3x3 matrix is tiny; OpenCV computes it on CPU, while the costly
        # pixel resampling below happens on Intel XPU through torch.grid_sample.
        matrix = cv2.getPerspectiveTransform(rect.astype(np.float32), dst)
        if image_tensor is None:
            image_tensor = bgr_numpy_to_tensor(image, DEVICE)
        elif image_tensor.device != DEVICE:
            image_tensor = image_tensor.to(DEVICE)

        with torch.inference_mode():
            crop_tensor = _xpu_warp_perspective(
                image_tensor, matrix, max_width, max_height
            )
    except Exception as exc:
        info['message'] = f'XPU perspective transform failed: {exc}'
        return None, info

    if crop_tensor is None or crop_tensor.numel() == 0:
        info['message'] = 'Perspective transform returned an empty crop.'
        return None, info

    if auto_rotate_landscape and crop_tensor.shape[-2] > crop_tensor.shape[-1]:
        # k=-1 is a 90-degree clockwise rotation over H,W.
        crop_tensor = torch.rot90(crop_tensor, k=-1, dims=(-2, -1)).contiguous()
        info['auto_rotated_landscape'] = True
    else:
        info['auto_rotated_landscape'] = False

    final_h = int(crop_tensor.shape[-2])
    final_w = int(crop_tensor.shape[-1])
    if final_w < min_crop_width or final_h < min_crop_height:
        info['message'] = f'Final crop too small after transform: {final_w}x{final_h}.'
        return None, info

    info['status'] = 'success'
    info['message'] = 'Intel XPU perspective crop created successfully.'
    info['final_width'] = final_w
    info['final_height'] = final_h
    return crop_tensor, info


def extract_plate_from_obb(image, points,
                           min_quad_area=MIN_QUAD_AREA,
                           min_crop_width=MIN_CROP_WIDTH,
                           min_crop_height=MIN_CROP_HEIGHT,
                           auto_rotate_landscape=AUTO_ROTATE_LANDSCAPE):
    """Preview-friendly wrapper that returns the rectified plate as NumPy BGR."""
    crop_tensor, info = extract_plate_tensor_from_obb(
        image,
        points,
        image_tensor=None,
        min_quad_area=min_quad_area,
        min_crop_width=min_crop_width,
        min_crop_height=min_crop_height,
        auto_rotate_landscape=auto_rotate_landscape,
    )
    if crop_tensor is None:
        return None, info
    return tensor_to_bgr_numpy(crop_tensor), info

## 9. Visualize Original OBB Labels — Stage B

This cell **does not save or process the full dataset**. It randomly selects valid annotations and shows:

- original image with the OBB polygon,
- ordered corner points labeled `TL`, `TR`, `BR`, `BL`,
- the final perspective-rectified crop beside it.

In [ ]:
def collect_preview_records(n=8, seed=RANDOM_SEED, max_images_to_scan=None):
    """Collect up to n valid OBB crops without saving anything."""
    image_files, label_files, _ = discover_input_files()
    image_index = build_stem_index(image_files)
    label_index = build_stem_index(label_files)

    matched = [stem for stem in sorted(set(image_index) & set(label_index))
               if len(image_index[stem]) == 1]
    rng = random.Random(seed)
    rng.shuffle(matched)

    records = []
    scanned = 0
    for stem in matched:
        if len(records) >= n:
            break
        if max_images_to_scan is not None and scanned >= max_images_to_scan:
            break
        scanned += 1

        image_path = image_index[stem][0]
        label_path = label_index[stem][0]
        image = read_image(image_path)
        if image is None:
            continue
        h, w = image.shape[:2]
        annotations, _ = parse_yolo_obb_label(label_path, w, h)
        if not annotations:
            continue

        annotation_order = list(range(len(annotations)))
        rng.shuffle(annotation_order)
        for idx in annotation_order:
            ann = annotations[idx]
            crop, crop_info = extract_plate_from_obb(image, ann['points'])
            if crop is None:
                continue
            records.append({
                'image_path': image_path,
                'label_path': label_path,
                'image': image,
                'annotation_index': idx + 1,
                'class_id': ann['class_id'],
                'points': ann['points'],
                'ordered_points': crop_info['ordered_points'],
                'crop': crop,
                'crop_info': crop_info,
            })
            # Use at most one annotation per source image so the preview spans different images.
            break

    return records


def show_obb_preview(records):
    if not records:
        print('No valid preview samples were found. Check validation messages and dataset paths.')
        return

    fig, axes = plt.subplots(len(records), 2, figsize=(16, max(4, 4 * len(records))))
    if len(records) == 1:
        axes = np.asarray([axes])

    labels = ['TL', 'TR', 'BR', 'BL']
    for row, rec in enumerate(records):
        image_rgb = cv2.cvtColor(rec['image'], cv2.COLOR_BGR2RGB)
        crop_rgb = cv2.cvtColor(rec['crop'], cv2.COLOR_BGR2RGB)
        ordered = rec['ordered_points']
        closed = np.vstack([ordered, ordered[0]])

        ax = axes[row, 0]
        ax.imshow(image_rgb)
        ax.plot(closed[:, 0], closed[:, 1], linewidth=2)
        ax.scatter(ordered[:, 0], ordered[:, 1], s=55)
        for point, label in zip(ordered, labels):
            ax.text(float(point[0]) + 4, float(point[1]) - 4, label,
                    fontsize=10, weight='bold',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
        ax.set_title(
            f"{rec['image_path'].name} | annotation {rec['annotation_index']} | class {rec['class_id']}"
        )
        ax.axis('off')

        ax = axes[row, 1]
        ax.imshow(crop_rgb)
        ax.set_title(
            f"Perspective crop: {rec['crop'].shape[1]} x {rec['crop'].shape[0]}"
        )
        ax.axis('off')

    plt.tight_layout()
    plt.show()


OBB_PREVIEW_RECORDS = collect_preview_records(OBB_PREVIEW_COUNT, RANDOM_SEED)
print(f'Valid OBB preview samples collected: {len(OBB_PREVIEW_RECORDS)}')
show_obb_preview(OBB_PREVIEW_RECORDS)

## 10. Preview Cropped License Plates

A clean-crop-only view makes it easier to inspect rectification quality without overlay graphics.

In [ ]:
def show_clean_crop_preview(records, max_items=CROP_PREVIEW_COUNT):
    records = records[:max_items]
    if not records:
        print('No valid crop previews available.')
        return

    cols = 2
    rows = math.ceil(len(records) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 4 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax in axes:
        ax.axis('off')

    for ax, rec in zip(axes, records):
        ax.imshow(cv2.cvtColor(rec['crop'], cv2.COLOR_BGR2RGB))
        ax.set_title(
            f"{rec['image_path'].name} | crop {rec['crop'].shape[1]}x{rec['crop'].shape[0]}"
        )
        ax.axis('off')

    plt.tight_layout()
    plt.show()


show_clean_crop_preview(OBB_PREVIEW_RECORDS)

## 11. Motion Blur Function — Intel XPU

The motion PSF is created on the CPU (it is tiny), moved to the selected device, and the image convolution runs with **PyTorch `conv2d` on Intel XPU**. Kernel size is automatically reduced for small crops; spatial dimensions are preserved.


In [ ]:
def _largest_valid_odd_size(requested_size, min_dimension):
    """Return an odd kernel size >=1 that does not exceed the smallest image dimension."""
    requested = max(1, int(round(requested_size)))
    max_allowed = max(1, int(min_dimension))
    if max_allowed % 2 == 0:
        max_allowed -= 1
    max_allowed = max(1, max_allowed)

    used = min(requested, max_allowed)
    if used % 2 == 0:
        used -= 1
    return max(1, used)


def create_motion_kernel(kernel_size, angle_degrees):
    """Create and normalize a rotated linear motion-blur kernel."""
    k = int(kernel_size)
    if k < 1 or k % 2 == 0:
        raise ValueError('Motion kernel size must be a positive odd integer.')

    # Creating this tiny PSF on CPU is negligible; convolution is on Intel XPU.
    kernel = np.zeros((k, k), dtype=np.float32)
    kernel[k // 2, :] = 1.0

    center = ((k - 1) / 2.0, (k - 1) / 2.0)
    matrix = cv2.getRotationMatrix2D(center, float(angle_degrees), 1.0)
    kernel = cv2.warpAffine(
        kernel, matrix, (k, k), flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0,
    )
    kernel = np.maximum(kernel, 0.0)
    total = float(kernel.sum())
    if total <= 0:
        raise ValueError('Motion kernel sum is zero after rotation.')
    kernel /= total
    return kernel


def _apply_kernel_tensor(image_tensor, kernel_2d):
    """Apply one 2D kernel independently to every image channel on the tensor device."""
    if image_tensor.ndim != 4 or image_tensor.shape[0] != 1:
        raise ValueError('Expected image tensor shape (1,C,H,W).')

    k_h, k_w = int(kernel_2d.shape[-2]), int(kernel_2d.shape[-1])
    if k_h != k_w or k_h % 2 == 0:
        raise ValueError('Kernel must be square with an odd size.')

    pad = k_h // 2
    if pad == 0:
        return image_tensor.clone()

    channels = int(image_tensor.shape[1])
    weight = kernel_2d.reshape(1, 1, k_h, k_w).to(
        device=image_tensor.device, dtype=image_tensor.dtype
    )
    weight = weight.repeat(channels, 1, 1, 1)

    padded = F.pad(image_tensor, (pad, pad, pad, pad), mode='reflect')
    return F.conv2d(padded, weight, bias=None, stride=1, padding=0, groups=channels)


def apply_motion_blur_tensor(image_tensor, kernel_size, angle_degrees):
    """Intel-XPU-capable motion blur. Returns tensor, actual kernel size, adjustment message."""
    if image_tensor is None or image_tensor.numel() == 0:
        raise ValueError('Cannot blur an empty tensor.')

    h, w = int(image_tensor.shape[-2]), int(image_tensor.shape[-1])
    used = _largest_valid_odd_size(kernel_size, min(h, w))
    adjustment = ''
    if used != int(kernel_size):
        adjustment = f'Motion kernel adjusted from {kernel_size} to {used} for {w}x{h} crop.'

    if used == 1:
        return image_tensor.clone(), used, adjustment or 'Motion kernel is 1; image is unchanged.'

    kernel_np = create_motion_kernel(used, angle_degrees)
    kernel_t = torch.as_tensor(kernel_np, device=image_tensor.device, dtype=image_tensor.dtype)
    with torch.inference_mode():
        blurred = _apply_kernel_tensor(image_tensor, kernel_t)

    if tuple(blurred.shape) != tuple(image_tensor.shape):
        raise RuntimeError('Motion blur unexpectedly changed tensor shape.')
    return blurred, used, adjustment


def apply_motion_blur(image, kernel_size, angle_degrees):
    """Preview-friendly NumPy wrapper around the Intel XPU motion-blur implementation."""
    if image is None or image.size == 0:
        raise ValueError('Cannot blur an empty image.')
    tensor = bgr_numpy_to_tensor(image, DEVICE)
    blurred_t, used, adjustment = apply_motion_blur_tensor(tensor, kernel_size, angle_degrees)
    blurred = tensor_to_bgr_numpy(blurred_t)
    assert blurred.shape == image.shape
    return blurred, used, adjustment

## 12. Defocus Blur Function — Intel XPU

Defocus blur uses a normalized circular/disk point-spread function. The PSF and the convolution are handled as PyTorch tensors, and the expensive convolution runs on **Intel XPU**. Gaussian blur is not substituted for defocus blur.


In [ ]:
def create_defocus_kernel_tensor(radius, device=DEVICE, dtype=torch.float32):
    """Create a normalized circular/disk PSF directly as a torch tensor."""
    r = int(round(radius))
    if r < 0:
        raise ValueError('Defocus radius cannot be negative.')
    if r == 0:
        return torch.ones((1, 1), device=device, dtype=dtype)

    coords = torch.arange(-r, r + 1, device=device, dtype=dtype)
    yy, xx = torch.meshgrid(coords, coords, indexing='ij')
    kernel = ((xx * xx + yy * yy) <= float(r * r)).to(dtype)
    total = kernel.sum()
    if float(total.item()) <= 0:
        raise ValueError('Defocus kernel sum is zero.')
    return kernel / total


def create_defocus_kernel(radius):
    """Compatibility/helper function returning the disk PSF as a NumPy array."""
    return create_defocus_kernel_tensor(radius, device=torch.device('cpu')).numpy()


def _largest_valid_defocus_radius(requested_radius, min_dimension):
    requested = max(0, int(round(requested_radius)))
    max_radius = max(0, (int(min_dimension) - 1) // 2)
    return min(requested, max_radius)


def apply_defocus_blur_tensor(image_tensor, radius):
    """Intel-XPU-capable disk-PSF defocus blur. Returns tensor, actual radius, message."""
    if image_tensor is None or image_tensor.numel() == 0:
        raise ValueError('Cannot blur an empty tensor.')

    h, w = int(image_tensor.shape[-2]), int(image_tensor.shape[-1])
    used = _largest_valid_defocus_radius(radius, min(h, w))
    adjustment = ''
    if used != int(radius):
        adjustment = f'Defocus radius adjusted from {radius} to {used} for {w}x{h} crop.'

    if used == 0:
        return image_tensor.clone(), used, adjustment or 'Defocus radius is 0; image is unchanged.'

    kernel_t = create_defocus_kernel_tensor(
        used, device=image_tensor.device, dtype=image_tensor.dtype
    )
    with torch.inference_mode():
        blurred = _apply_kernel_tensor(image_tensor, kernel_t)

    if tuple(blurred.shape) != tuple(image_tensor.shape):
        raise RuntimeError('Defocus blur unexpectedly changed tensor shape.')
    return blurred, used, adjustment


def apply_defocus_blur(image, radius):
    """Preview-friendly NumPy wrapper around the Intel XPU defocus implementation."""
    if image is None or image.size == 0:
        raise ValueError('Cannot blur an empty image.')
    tensor = bgr_numpy_to_tensor(image, DEVICE)
    blurred_t, used, adjustment = apply_defocus_blur_tensor(tensor, radius)
    blurred = tensor_to_bgr_numpy(blurred_t)
    assert blurred.shape == image.shape
    return blurred, used, adjustment


def deterministic_rng_for_name(name, base_seed=RANDOM_SEED):
    """Return a stable RNG for a filename, independent of processing/skipping order."""
    digest = hashlib.sha256(f'{base_seed}|{name}'.encode('utf-8')).digest()
    seed = int.from_bytes(digest[:8], 'little', signed=False)
    return np.random.default_rng(seed)


def deterministic_motion_angle(name,
                               angle_min=MOTION_ANGLE_MIN,
                               angle_max=MOTION_ANGLE_MAX,
                               base_seed=RANDOM_SEED):
    rng = deterministic_rng_for_name(name, base_seed)
    return float(rng.uniform(float(angle_min), float(angle_max)))

## 13. Blur Parameter Settings

The values below are still the only blur-strength controls you normally need to tune. Preview functions call the Intel XPU implementations automatically because `DEVICE` is set globally.

In [ ]:
print('Current blur parameters in memory')
print('  MOTION_KERNEL_SIZE =', MOTION_KERNEL_SIZE)
print('  MOTION_ANGLE_MIN   =', MOTION_ANGLE_MIN)
print('  MOTION_ANGLE_MAX   =', MOTION_ANGLE_MAX)
print('  DEFOCUS_RADIUS     =', DEFOCUS_RADIUS)
print('  RANDOM_SEED        =', RANDOM_SEED)
print('  COMPUTE_DEVICE     =', DEVICE)
if DEVICE.type == 'xpu':
    print('  GPU NAME           =', torch.xpu.get_device_properties(XPU_INDEX).name)

## 14. Random Blur Preview — Stage C

For each preview plate, the three columns show **Clean | Motion Blur | Defocus Blur**. Nothing is saved. Motion and defocus are both generated independently from the same clean crop.

In [ ]:
def show_blur_preview(records, n=BLUR_PREVIEW_COUNT):
    if not records:
        print('No valid clean crops available for blur preview.')
        return

    rng = random.Random(RANDOM_SEED)
    chosen = list(records)
    rng.shuffle(chosen)
    chosen = chosen[:min(n, len(chosen))]

    fig, axes = plt.subplots(len(chosen), 3, figsize=(18, max(4, 4 * len(chosen))))
    if len(chosen) == 1:
        axes = np.asarray([axes])

    for row, rec in enumerate(chosen):
        clean = rec['crop']
        preview_name = f"{rec['image_path'].stem}_ann{rec['annotation_index']:02d}{OUTPUT_EXTENSION}"
        angle = deterministic_motion_angle(preview_name)
        motion, used_k, motion_note = apply_motion_blur(clean, MOTION_KERNEL_SIZE, angle)
        defocus, used_r, defocus_note = apply_defocus_blur(clean, DEFOCUS_RADIUS)

        assert clean.shape == motion.shape == defocus.shape

        images = [clean, motion, defocus]
        titles = [
            f"Clean\n{clean.shape[1]}x{clean.shape[0]}",
            f"Motion blur\nk={used_k}, angle={angle:.1f}°",
            f"Defocus blur\nradius={used_r}",
        ]
        for col in range(3):
            axes[row, col].imshow(cv2.cvtColor(images[col], cv2.COLOR_BGR2RGB))
            axes[row, col].set_title(titles[col])
            axes[row, col].axis('off')

        if motion_note:
            print(preview_name, '-', motion_note)
        if defocus_note:
            print(preview_name, '-', defocus_note)

    plt.tight_layout()
    plt.show()


show_blur_preview(OBB_PREVIEW_RECORDS, BLUR_PREVIEW_COUNT)

## 15. Blur Strength Comparison

This visualization helps choose blur severity. It does **not** save images.

In [ ]:
MOTION_KERNEL_COMPARISON = [5, 9, 15, 21]
DEFOCUS_RADIUS_COMPARISON = [2, 3, 5, 7]


def show_blur_strength_comparison(records):
    if not records:
        print('No valid crop available for comparison.')
        return

    clean = records[0]['crop']
    sample_name = f"{records[0]['image_path'].stem}_comparison{OUTPUT_EXTENSION}"
    angle = deterministic_motion_angle(sample_name)

    # Motion comparison
    motion_images = [clean]
    motion_titles = ['Clean']
    for k in MOTION_KERNEL_COMPARISON:
        blurred, used_k, note = apply_motion_blur(clean, k, angle)
        motion_images.append(blurred)
        motion_titles.append(f'Motion k={used_k}\nangle={angle:.1f}°')
        if note:
            print('Motion comparison:', note)

    fig, axes = plt.subplots(1, len(motion_images), figsize=(4 * len(motion_images), 4))
    for ax, image, title in zip(np.atleast_1d(axes), motion_images, motion_titles):
        ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    plt.suptitle('Motion blur strength comparison')
    plt.tight_layout()
    plt.show()

    # Defocus comparison
    defocus_images = [clean]
    defocus_titles = ['Clean']
    for r in DEFOCUS_RADIUS_COMPARISON:
        blurred, used_r, note = apply_defocus_blur(clean, r)
        defocus_images.append(blurred)
        defocus_titles.append(f'Defocus radius={used_r}')
        if note:
            print('Defocus comparison:', note)

    fig, axes = plt.subplots(1, len(defocus_images), figsize=(4 * len(defocus_images), 4))
    for ax, image, title in zip(np.atleast_1d(axes), defocus_images, defocus_titles):
        ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    plt.suptitle('Defocus blur strength comparison')
    plt.tight_layout()
    plt.show()


show_blur_strength_comparison(OBB_PREVIEW_RECORDS)

## 16. FULL DATASET GENERATION — Intel XPU

> **RUN THIS SECTION ONLY AFTER YOU ARE SATISFIED WITH THE OBB CROP AND BLUR PREVIEWS ABOVE.**

For each source image, the image is decoded on CPU and uploaded to the Intel Arc GPU once. All valid OBB crops from that source reuse the same XPU tensor. Perspective sampling and both blur convolutions run on XPU; final images are copied back to CPU only for JPEG encoding/saving.


In [ ]:
def make_plate_filename(image_path: Path, annotation_position: int, valid_annotation_count: int):
    """Keep the source stem; add _lpNN only when multiple valid OBBs are present."""
    if valid_annotation_count == 1:
        return f'{image_path.stem}{OUTPUT_EXTENSION}'
    return f'{image_path.stem}_lp{annotation_position:02d}{OUTPUT_EXTENSION}'


def encode_jpeg(image, quality=JPEG_QUALITY):
    """Encode once in memory at high JPEG quality."""
    ok, encoded = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    if not ok:
        raise IOError('OpenCV JPEG encoding failed.')
    return encoded


def write_triplet_atomically(clean, motion, defocus, clean_path, motion_path, defocus_path):
    """Encode all three first, write temp files, then replace final paths."""
    encoded_items = [
        (encode_jpeg(clean), clean_path),
        (encode_jpeg(motion), motion_path),
        (encode_jpeg(defocus), defocus_path),
    ]
    temp_paths = []
    try:
        for encoded, final_path in encoded_items:
            tmp = final_path.with_name(final_path.name + '.tmp')
            encoded.tofile(str(tmp))
            temp_paths.append(tmp)
        for tmp, (_, final_path) in zip(temp_paths, encoded_items):
            tmp.replace(final_path)
        return True, ''
    except Exception as exc:
        for tmp in temp_paths:
            try:
                if tmp.exists():
                    tmp.unlink()
            except Exception:
                pass
        return False, str(exc)


def output_triplet_state(plate_name):
    paths = (
        CLEAN_OUTPUT / plate_name,
        MOTION_OUTPUT / plate_name,
        DEFOCUS_OUTPUT / plate_name,
    )
    exists = tuple(p.exists() for p in paths)
    if all(exists):
        return 'complete', paths
    if any(exists):
        return 'partial', paths
    return 'missing', paths


def generate_full_dataset():
    image_files, label_files, unsupported_files = discover_input_files()
    image_index = build_stem_index(image_files)
    label_index = build_stem_index(label_files)

    for folder in [OUTPUT_ROOT, CLEAN_OUTPUT, MOTION_OUTPUT, DEFOCUS_OUTPUT]:
        folder.mkdir(parents=True, exist_ok=True)

    report_rows = []
    counters = Counter()

    for path in unsupported_files:
        report_rows.append({
            'original_image': path.name, 'label_file': '', 'plate_filename': '',
            'annotation_index': '', 'status': 'unsupported_file',
            'clean_width': '', 'clean_height': '', 'motion_kernel': '',
            'motion_angle': '', 'defocus_radius': '',
            'message': f'Unsupported file type: {path.suffix}',
        })
        counters['unsupported_files'] += 1

    for stem in sorted(set(label_index) - set(image_index)):
        for label_path in label_index[stem]:
            report_rows.append({
                'original_image': '', 'label_file': label_path.name, 'plate_filename': '',
                'annotation_index': '', 'status': 'missing_image',
                'clean_width': '', 'clean_height': '', 'motion_kernel': '',
                'motion_angle': '', 'defocus_radius': '',
                'message': 'Label has no corresponding supported image.',
            })
            counters['missing_images'] += 1

    print('=' * 72)
    print('FULL DATASET GENERATION — INTEL XPU')
    print('=' * 72)
    print('OVERWRITE_EXISTING:', OVERWRITE_EXISTING)
    print('Compute device:', DEVICE)
    if DEVICE.type == 'xpu':
        print('GPU:', torch.xpu.get_device_properties(XPU_INDEX).name)
    print('Current motion kernel:', MOTION_KERNEL_SIZE)
    print('Current motion angle range:', (MOTION_ANGLE_MIN, MOTION_ANGLE_MAX))
    print('Current defocus radius:', DEFOCUS_RADIUS)
    print()

    for image_path in tqdm(image_files, desc='Processing source images on Intel XPU'):
        counters['original_images_seen'] += 1
        stem = image_path.stem

        if len(image_index[stem]) > 1:
            counters['skipped_images'] += 1
            report_rows.append({
                'original_image': image_path.name,
                'label_file': f'{stem}.txt' if stem in label_index else '',
                'plate_filename': '', 'annotation_index': '',
                'status': 'duplicate_image_stem',
                'clean_width': '', 'clean_height': '', 'motion_kernel': '',
                'motion_angle': '', 'defocus_radius': '',
                'message': 'Multiple source images share this stem; skipped to avoid output collisions.',
            })
            continue

        label_paths = label_index.get(stem, [])
        if not label_paths:
            counters['missing_labels'] += 1
            counters['skipped_images'] += 1
            report_rows.append({
                'original_image': image_path.name, 'label_file': '', 'plate_filename': '',
                'annotation_index': '', 'status': 'missing_label',
                'clean_width': '', 'clean_height': '', 'motion_kernel': '',
                'motion_angle': '', 'defocus_radius': '',
                'message': 'No corresponding .txt label file.',
            })
            continue

        label_path = label_paths[0]
        image = read_image(image_path)
        if image is None:
            counters['failed_image_reads'] += 1
            counters['skipped_images'] += 1
            report_rows.append({
                'original_image': image_path.name, 'label_file': label_path.name,
                'plate_filename': '', 'annotation_index': '',
                'status': 'image_read_failed', 'clean_width': '', 'clean_height': '',
                'motion_kernel': '', 'motion_angle': '', 'defocus_radius': '',
                'message': 'OpenCV could not decode the image.',
            })
            continue

        h, w = image.shape[:2]
        annotations, issues = parse_yolo_obb_label(label_path, w, h)

        for issue in issues:
            status = issue['status']
            if status == 'invalid_annotation':
                counters['invalid_annotations'] += 1
            elif status == 'empty_label':
                counters['empty_labels'] += 1
            elif status == 'label_read_failed':
                counters['label_read_failures'] += 1
            elif status == 'coordinate_clipped':
                counters['coordinate_clips'] += 1

            report_rows.append({
                'original_image': image_path.name, 'label_file': label_path.name,
                'plate_filename': '', 'annotation_index': issue.get('line') or '',
                'status': status, 'clean_width': '', 'clean_height': '',
                'motion_kernel': '', 'motion_angle': '', 'defocus_radius': '',
                'message': issue['message'],
            })

        if not annotations:
            counters['skipped_images'] += 1
            continue

        # Upload the source image once. Multiple OBBs reuse the same XPU tensor.
        try:
            source_tensor = bgr_numpy_to_tensor(image, DEVICE)
        except Exception as exc:
            counters['xpu_upload_failures'] += 1
            counters['skipped_images'] += 1
            report_rows.append({
                'original_image': image_path.name, 'label_file': label_path.name,
                'plate_filename': '', 'annotation_index': '',
                'status': 'xpu_upload_failed', 'clean_width': '', 'clean_height': '',
                'motion_kernel': '', 'motion_angle': '', 'defocus_radius': '',
                'message': str(exc),
            })
            continue

        any_success_for_image = False
        valid_count = len(annotations)

        for valid_position, ann in enumerate(annotations, start=1):
            plate_name = make_plate_filename(image_path, valid_position, valid_count)
            clean_path = CLEAN_OUTPUT / plate_name
            motion_path = MOTION_OUTPUT / plate_name
            defocus_path = DEFOCUS_OUTPUT / plate_name

            state, paths = output_triplet_state(plate_name)
            if state == 'complete' and not OVERWRITE_EXISTING:
                existing = read_image(clean_path)
                existing_h = int(existing.shape[0]) if existing is not None else ''
                existing_w = int(existing.shape[1]) if existing is not None else ''
                counters['existing_complete'] += 1
                any_success_for_image = True
                report_rows.append({
                    'original_image': image_path.name, 'label_file': label_path.name,
                    'plate_filename': plate_name, 'annotation_index': valid_position,
                    'status': 'already_exists', 'clean_width': existing_w,
                    'clean_height': existing_h, 'motion_kernel': '', 'motion_angle': '',
                    'defocus_radius': '',
                    'message': 'Complete output triplet already exists; skipped because OVERWRITE_EXISTING=False.',
                })
                continue

            if state == 'partial' and not OVERWRITE_EXISTING:
                counters['partial_existing_skips'] += 1
                report_rows.append({
                    'original_image': image_path.name, 'label_file': label_path.name,
                    'plate_filename': plate_name, 'annotation_index': valid_position,
                    'status': 'partial_output_exists', 'clean_width': '', 'clean_height': '',
                    'motion_kernel': '', 'motion_angle': '', 'defocus_radius': '',
                    'message': (
                        'Only part of the clean/motion/defocus triplet exists. '
                        'Skipped to avoid silently mixing old and new outputs. '
                        'Set OVERWRITE_EXISTING=True to regenerate all three.'
                    ),
                })
                continue

            clean_t, crop_info = extract_plate_tensor_from_obb(
                image, ann['points'], image_tensor=source_tensor
            )
            if clean_t is None:
                counters['invalid_crops'] += 1
                report_rows.append({
                    'original_image': image_path.name, 'label_file': label_path.name,
                    'plate_filename': plate_name, 'annotation_index': valid_position,
                    'status': 'invalid_crop', 'clean_width': '', 'clean_height': '',
                    'motion_kernel': '', 'motion_angle': '', 'defocus_radius': '',
                    'message': crop_info.get('message', 'Invalid crop.'),
                })
                continue

            angle = deterministic_motion_angle(plate_name)
            try:
                with torch.inference_mode():
                    motion_t, used_k, motion_note = apply_motion_blur_tensor(
                        clean_t, MOTION_KERNEL_SIZE, angle
                    )
                    defocus_t, used_r, defocus_note = apply_defocus_blur_tensor(
                        clean_t, DEFOCUS_RADIUS
                    )

                if tuple(clean_t.shape) != tuple(motion_t.shape) or tuple(clean_t.shape) != tuple(defocus_t.shape):
                    raise RuntimeError('Clean/motion/defocus tensor shapes do not match.')

                # Copy to CPU only after all Intel XPU operations are complete.
                clean = tensor_to_bgr_numpy(clean_t)
                motion = tensor_to_bgr_numpy(motion_t)
                defocus = tensor_to_bgr_numpy(defocus_t)
                assert clean.shape == motion.shape == defocus.shape
            except Exception as exc:
                counters['blur_failures'] += 1
                report_rows.append({
                    'original_image': image_path.name, 'label_file': label_path.name,
                    'plate_filename': plate_name, 'annotation_index': valid_position,
                    'status': 'blur_failed',
                    'clean_width': int(clean_t.shape[-1]),
                    'clean_height': int(clean_t.shape[-2]),
                    'motion_kernel': '', 'motion_angle': angle, 'defocus_radius': '',
                    'message': str(exc),
                })
                continue

            adjustment_messages = [m for m in [motion_note, defocus_note] if m]
            if adjustment_messages:
                counters['automatic_blur_adjustments'] += 1
                print(f'\n[adjustment] {plate_name}: ' + ' | '.join(adjustment_messages))

            ok, save_error = write_triplet_atomically(
                clean, motion, defocus, clean_path, motion_path, defocus_path
            )
            if not ok:
                counters['save_failures'] += 1
                report_rows.append({
                    'original_image': image_path.name, 'label_file': label_path.name,
                    'plate_filename': plate_name, 'annotation_index': valid_position,
                    'status': 'save_failed', 'clean_width': int(clean.shape[1]),
                    'clean_height': int(clean.shape[0]), 'motion_kernel': used_k,
                    'motion_angle': angle, 'defocus_radius': used_r,
                    'message': save_error,
                })
                continue

            counters['successful'] += 1
            counters['xpu_processed_plates'] += 1
            any_success_for_image = True
            report_rows.append({
                'original_image': image_path.name, 'label_file': label_path.name,
                'plate_filename': plate_name, 'annotation_index': valid_position,
                'status': 'success', 'clean_width': int(clean.shape[1]),
                'clean_height': int(clean.shape[0]), 'motion_kernel': used_k,
                'motion_angle': round(angle, 4), 'defocus_radius': used_r,
                'message': ' | '.join(adjustment_messages),
            })

        if not any_success_for_image:
            counters['images_without_successful_output'] += 1

        # Drop the source tensor reference before moving to the next source image.
        del source_tensor

    if DEVICE.type == 'xpu':
        torch.xpu.synchronize(DEVICE)

    report_columns = [
        'original_image', 'label_file', 'plate_filename', 'annotation_index',
        'status', 'clean_width', 'clean_height', 'motion_kernel',
        'motion_angle', 'defocus_radius', 'message'
    ]
    report_df = pd.DataFrame(report_rows, columns=report_columns)
    report_df.to_csv(REPORT_PATH, index=False, encoding='utf-8-sig')

    print('\n' + '=' * 72)
    print('FULL GENERATION RUN SUMMARY')
    print('=' * 72)
    print('Compute device                      :', DEVICE)
    if DEVICE.type == 'xpu':
        print('GPU                                 :', torch.xpu.get_device_properties(XPU_INDEX).name)
    for key in [
        'original_images_seen', 'successful', 'xpu_processed_plates',
        'existing_complete', 'skipped_images', 'missing_labels', 'missing_images',
        'invalid_annotations', 'empty_labels', 'failed_image_reads', 'invalid_crops',
        'xpu_upload_failures', 'blur_failures', 'save_failures',
        'partial_existing_skips', 'automatic_blur_adjustments', 'unsupported_files'
    ]:
        print(f'{key:36s}: {counters[key]}')
    print(f'Processing report: {REPORT_PATH}')
    print('=' * 72)

    return counters, report_df


# RUN THIS CELL ONLY AFTER YOU ARE SATISFIED WITH THE BLUR PREVIEW ABOVE.
FULL_RUN_COUNTERS, PROCESSING_REPORT = generate_full_dataset()

## 17. Filename Consistency Test

Checks that the three output folders contain the same filenames and reports missing/extra files.

In [ ]:
def generated_name_set(folder: Path):
    return {
        p.name for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() == OUTPUT_EXTENSION.lower()
    }


def check_filename_consistency(verbose=True):
    clean_names = generated_name_set(CLEAN_OUTPUT)
    motion_names = generated_name_set(MOTION_OUTPUT)
    defocus_names = generated_name_set(DEFOCUS_OUTPUT)

    matching = clean_names & motion_names & defocus_names
    missing_motion = clean_names - motion_names
    missing_defocus = clean_names - defocus_names
    extra_motion = motion_names - clean_names
    extra_defocus = defocus_names - clean_names
    passed = clean_names == motion_names == defocus_names

    if verbose:
        print('=' * 60)
        print('DATASET CONSISTENCY CHECK')
        print('=' * 60)
        print(f'Clean images:         {len(clean_names)}')
        print(f'Motion blur images:   {len(motion_names)}')
        print(f'Defocus blur images:  {len(defocus_names)}')
        print()
        print(f'Matching filenames:   {len(matching)}')
        print(f'Missing motion:       {len(missing_motion)}')
        print(f'Missing defocus:      {len(missing_defocus)}')
        print(f'Extra motion:         {len(extra_motion)}')
        print(f'Extra defocus:        {len(extra_defocus)}')
        print()
        print('Filename consistency:', 'PASSED' if passed else 'FAILED')
        print('=' * 60)

        if not passed:
            if missing_motion:
                print('Missing motion examples:', sorted(missing_motion)[:20])
            if missing_defocus:
                print('Missing defocus examples:', sorted(missing_defocus)[:20])
            if extra_motion:
                print('Extra motion examples:', sorted(extra_motion)[:20])
            if extra_defocus:
                print('Extra defocus examples:', sorted(extra_defocus)[:20])

    return {
        'passed': passed,
        'clean_names': clean_names,
        'motion_names': motion_names,
        'defocus_names': defocus_names,
        'matching_names': matching,
        'missing_motion': missing_motion,
        'missing_defocus': missing_defocus,
        'extra_motion': extra_motion,
        'extra_defocus': extra_defocus,
    }


FILENAME_CHECK = check_filename_consistency()

## 18. Dataset Statistics and Shape Integrity

This section reports clean-crop dimensions and then verifies that every clean/motion/defocus triplet is readable and has identical shape.

In [ ]:
def collect_clean_plate_statistics():
    files = sorted(
        [p for p in CLEAN_OUTPUT.iterdir() if p.is_file() and p.suffix.lower() == OUTPUT_EXTENSION.lower()],
        key=lambda p: p.name.lower()
    )
    records = []
    unreadable = []

    for path in tqdm(files, desc='Reading clean plates for statistics'):
        image = read_image(path)
        if image is None:
            unreadable.append(path.name)
            continue
        h, w = image.shape[:2]
        records.append({
            'filename': path.name,
            'width': w,
            'height': h,
            'area': w * h,
            'aspect_ratio': w / h if h else np.nan,
        })

    df = pd.DataFrame(records)
    print('=' * 64)
    print('CLEAN PLATE STATISTICS')
    print('=' * 64)
    print(f'Number of readable plates: {len(df)}')
    print(f'Unreadable clean plates:    {len(unreadable)}')

    if not df.empty:
        smallest = df.loc[df['area'].idxmin()]
        largest = df.loc[df['area'].idxmax()]
        print(f"Minimum width:              {int(df['width'].min())}")
        print(f"Maximum width:              {int(df['width'].max())}")
        print(f"Average width:              {df['width'].mean():.2f}")
        print(f"Minimum height:             {int(df['height'].min())}")
        print(f"Maximum height:             {int(df['height'].max())}")
        print(f"Average height:             {df['height'].mean():.2f}")
        print(f"Average aspect ratio:       {df['aspect_ratio'].mean():.3f}")
        print(
            f"Smallest crop:              {smallest['filename']} "
            f"({int(smallest['width'])}x{int(smallest['height'])})"
        )
        print(
            f"Largest crop:               {largest['filename']} "
            f"({int(largest['width'])}x{int(largest['height'])})"
        )
    print('=' * 64)

    if unreadable:
        print('Unreadable examples:', unreadable[:20])

    return df


CLEAN_STATS = collect_clean_plate_statistics()

if not CLEAN_STATS.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(CLEAN_STATS['width'], bins=30)
    axes[0].set_title('Clean plate width distribution')
    axes[0].set_xlabel('Width (pixels)')
    axes[0].set_ylabel('Count')

    axes[1].hist(CLEAN_STATS['height'], bins=30)
    axes[1].set_title('Clean plate height distribution')
    axes[1].set_xlabel('Height (pixels)')
    axes[1].set_ylabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
def validate_output_integrity(max_failures_to_show=30):
    """Verify existence, readability, and shape equality for all three folders."""
    filename_check = check_filename_consistency(verbose=False)
    all_names = sorted(
        filename_check['clean_names'] |
        filename_check['motion_names'] |
        filename_check['defocus_names']
    )

    failures = []
    checked_triplets = 0

    for name in tqdm(all_names, desc='Validating output triplets'):
        paths = [CLEAN_OUTPUT / name, MOTION_OUTPUT / name, DEFOCUS_OUTPUT / name]
        labels = ['clean', 'motion', 'defocus']
        missing = [label for label, path in zip(labels, paths) if not path.exists()]
        if missing:
            failures.append((name, 'missing_files', ', '.join(missing)))
            continue

        images = [read_image(path) for path in paths]
        unreadable = [label for label, image in zip(labels, images) if image is None]
        if unreadable:
            failures.append((name, 'unreadable', ', '.join(unreadable)))
            continue

        shapes = [image.shape for image in images]
        if not (shapes[0] == shapes[1] == shapes[2]):
            failures.append((name, 'shape_mismatch', str(shapes)))
            continue

        checked_triplets += 1

    passed = filename_check['passed'] and not failures

    print('=' * 64)
    print('DATASET INTEGRITY CHECK')
    print('=' * 64)
    print(f'Complete readable shape-matched triplets: {checked_triplets}')
    print(f'Failures:                                 {len(failures)}')
    print('Dataset integrity check:', 'PASSED' if passed else 'FAILED')
    print('=' * 64)

    if failures:
        print(f'First {min(len(failures), max_failures_to_show)} failures:')
        for item in failures[:max_failures_to_show]:
            print(' ', item)

    return {'passed': passed, 'checked_triplets': checked_triplets, 'failures': failures}


INTEGRITY_CHECK = validate_output_integrity()

## 19. Random Final Visual Verification

Randomly loads matching filenames from all three output folders and shows the exact triplets side-by-side.

In [ ]:
def show_final_random_triplets(n=FINAL_PREVIEW_COUNT, seed=RANDOM_SEED):
    clean_names = generated_name_set(CLEAN_OUTPUT)
    motion_names = generated_name_set(MOTION_OUTPUT)
    defocus_names = generated_name_set(DEFOCUS_OUTPUT)
    common = sorted(clean_names & motion_names & defocus_names)

    if not common:
        print('No complete triplets available for final preview.')
        return

    rng = random.Random(seed)
    selected = rng.sample(common, k=min(n, len(common)))

    fig, axes = plt.subplots(len(selected), 3, figsize=(18, max(4, 4 * len(selected))))
    if len(selected) == 1:
        axes = np.asarray([axes])

    for row, name in enumerate(selected):
        images = [
            read_image(CLEAN_OUTPUT / name),
            read_image(MOTION_OUTPUT / name),
            read_image(DEFOCUS_OUTPUT / name),
        ]
        titles = [f'Clean\n{name}', 'Motion Blur', 'Defocus Blur']

        for col, (image, title) in enumerate(zip(images, titles)):
            ax = axes[row, col]
            if image is None:
                ax.text(0.5, 0.5, 'Unreadable', ha='center', va='center')
            else:
                ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            ax.set_title(title)
            ax.axis('off')

    plt.tight_layout()
    plt.show()


show_final_random_triplets()

## 20. Final Summary

This cell prints the final dataset report using the current output folders and, when available, the current processing report.

In [ ]:
def print_final_summary():
    image_files, label_files, _ = discover_input_files()
    filename_check = check_filename_consistency(verbose=False)

    report_df = None
    if REPORT_PATH.exists():
        try:
            report_df = pd.read_csv(REPORT_PATH)
        except Exception as exc:
            print('Warning: could not read processing report:', exc)

    status_counts = Counter()
    if report_df is not None and 'status' in report_df.columns:
        status_counts.update(report_df['status'].fillna('').astype(str))

    print('=' * 72)
    print('FINAL DATASET REPORT')
    print('=' * 72)
    print(f'Original images found:                  {len(image_files)}')
    print(f'Label files found:                      {len(label_files)}')
    print()
    print(f"Successfully generated this run:        {status_counts['success']}")
    print(f"Already-complete samples skipped:       {status_counts['already_exists']}")
    print(f"Missing labels:                         {status_counts['missing_label']}")
    print(f"Invalid annotation rows:                {status_counts['invalid_annotation']}")
    print(f"Unreadable images:                      {status_counts['image_read_failed']}")
    print(f"Invalid/tiny OBB crops:                 {status_counts['invalid_crop']}")
    print(f"Partial-output skips:                   {status_counts['partial_output_exists']}")
    print()
    print(f"Clean plates present:                   {len(filename_check['clean_names'])}")
    print(f"Motion-blurred plates present:          {len(filename_check['motion_names'])}")
    print(f"Defocus-blurred plates present:         {len(filename_check['defocus_names'])}")
    print()
    print('Filename consistency:')
    print('Clean == Motion == Defocus              ', 'PASSED' if filename_check['passed'] else 'FAILED')
    if 'INTEGRITY_CHECK' in globals():
        print('Readable + shape consistency             ', 'PASSED' if INTEGRITY_CHECK['passed'] else 'FAILED')
    print()
    print('Output directory:')
    print(OUTPUT_ROOT)
    print('Processing report:')
    print(REPORT_PATH)
    print()
    if filename_check['passed'] and (('INTEGRITY_CHECK' not in globals()) or INTEGRITY_CHECK['passed']):
        print('Processing/validation completed successfully for the files currently present.')
    else:
        print('Validation found mismatches. Review the printed failures before training.')
    print('=' * 72)


print_final_summary()

# How to Run This Notebook in VS Code

1. Put this notebook in your project folder.
2. Activate the same `.venv` you installed the XPU build of PyTorch into.
3. Install normal dependencies if needed:
   `pip install opencv-python numpy matplotlib tqdm pandas jupyter ipykernel`
4. Install/confirm Intel XPU PyTorch:
   `pip install torch torchvision --index-url https://download.pytorch.org/whl/xpu`
5. Select that `.venv` as the notebook kernel in VS Code.
6. Run the import cell and confirm `XPU available : True`.
7. Run the Intel XPU smoke test and confirm `Intel XPU smoke test: PASSED`.
8. Change `DATASET_ROOT` in Configuration. Keep `REQUIRE_XPU = True`.
9. Run Dataset Validation.
10. Run the annotation parser, point-ordering, and Intel-XPU perspective-extraction definitions.
11. Run the OBB visualization and clean-crop preview. **Do not continue if polygons/crops are wrong.**
12. Run the Intel-XPU motion/defocus function cells.
13. Run Random Blur Preview and Blur Strength Comparison.
14. Adjust `MOTION_KERNEL_SIZE`, angle range, and `DEFOCUS_RADIUS` if necessary; rerun only the settings + preview cells.
15. When satisfied, run **FULL DATASET GENERATION — Intel XPU**.
16. Run Filename Consistency Test, Dataset Statistics / Shape Integrity, Random Final Visual Verification, and Final Summary.

### How to confirm your Intel Arc A750 is being used

While the full-generation cell is running, open **Windows Task Manager → Performance → GPU**. Choose the Intel Arc A750 and view a Compute/Compute_0 graph. GPU usage may rise and fall because JPEG decoding/encoding and disk I/O remain CPU-side, and license-plate crops are relatively small.

You can also run this in a notebook cell at any time:

```python
print(torch.xpu.is_available())
print(torch.xpu.get_device_properties(0).name)
```

Expected for your PC:

```text
True
Intel(R) Arc(TM) A750 Graphics
```

### Important

Selecting a Jupyter kernel alone does not enable Intel GPU processing. The selected environment must contain the **XPU build of PyTorch**, and `torch.xpu.is_available()` must be `True`.
